In [0]:
# ============================================================
# Bronze — Source 01: RDS PostgreSQL
#
# Incremental load using watermark pattern.
# Only processes S3 files newer than last successful run.
# On first run: loads all historical data (watermark = epoch).
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=01_postgres/
# Target: bronze.postgres catalog in Unity Catalog
#
# Tables:
#   - bronze.postgres.orders
#   - bronze.postgres.customers
#   - bronze.postgres.payments
#   - bronze.postgres.order_items
#   - bronze.postgres.inventory
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "01_postgres"
TABLES = ["orders", "customers", "payments", "order_items", "inventory"]

In [0]:
# ============================================================
# Incremental load for all Postgres tables
#
# Uses file path date partitions to determine new files.
# Watermark stores last processed date.
# ============================================================

from pyspark.sql.functions import col, lit, max as spark_max
from pyspark.sql import functions as F

# Get watermark
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

total_rows = 0

for table in TABLES:
    path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/table={table}/"
    
    # Read all files — filter by file modification time
    df = spark.read.format("parquet") \
        .load(path) \
        .filter(col("_metadata.file_modification_time") > lit(watermark))
    
    row_count = df.count()
    
    if row_count == 0:
        print(f"[{table}] No new data — skipping")
        continue
    
    # Create schema if first run
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.postgres")
    
    # Append new rows to Bronze Delta table
    df.drop("_metadata").write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"bronze.postgres.{table}")
    
    total_rows += row_count
    print(f"✅ bronze.postgres.{table}: {row_count} new rows")

# Get latest file modification time across all files
latest_ts = spark.read.format("parquet") \
    .load(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/table=orders/") \
    .select(spark_max("_metadata.file_modification_time")).collect()[0][0]

if total_rows > 0:
    update_watermark(spark, SOURCE, latest_ts, total_rows)
    print(f"\n✅ {total_rows} total rows loaded, watermark updated to {latest_ts}")
else:
    print("\n⏭ No new data")

In [0]:
# Verify row counts in Bronze tables
print("=== Bronze.postgres table counts ===")
for table in TABLES:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM bronze.postgres.{table}").collect()[0]['cnt']
    print(f"  bronze.postgres.{table}: {count} rows")

print("\n=== Watermark state ===")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '01_postgres'").show()